In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os 
import sys 
import warnings
sys.path.append('..')
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt 
import numpy as np
import torch
import compress_pickle

import hippocampalseq as hse
import hippocampalseq.utils as hseu
import hippocampalseq.preprocessing as hsep
import hippocampalseq.analysis as hsea
import hippocampalseq.models as hsem
import hippocampalseq.plotting as hsepl

In [ ]:
results_dir = '/project/bioinformatics/WZhang_lab/shared/theta_momentum/10ms_theta_nonflat/'

def load_all(file_name, results_dir, track_type=["Linear", "Open"]):
    if not isinstance(track_type, list):
        track_type = [track_type]
    sessions = []
    for rat in os.listdir(results_dir):
        rat_dir = os.path.join(results_dir,rat)
        sess = {}
        if not os.path.isdir(rat_dir):
            continue
        for session in os.listdir(rat_dir):
            if session[:-1] not in track_type:
                continue
            sess[session] = compress_pickle.load(
                os.path.join(rat_dir, session, file_name)
            )
        sessions.append(sess)
    return sessions

rat_session_data = load_all('preprocessed_data.pkl.gz', results_dir, "Linear")
rat_session_models = load_all('model_theta_results.pkl.gz', results_dir, "Linear")

In [ ]:
rat_session_data

In [ ]:
def run_peak_posteriors(data: list, models: list):
    outputs = []
    for rat_session_dat,rat_session_mod in zip(data,models):
        session_models = {}
        for name in rat_session_dat.keys():
            track_type = name[:-1]
            for model_name,model_result in rat_session_mod[name]:
                theta_data = rat_session_dat[name]['theta_data']
                environment_size = rat_session_dat[name]['raw_data'].environment_size
                bin_size = rat_session_dat[name]['raw_data'].bin_size
                if model_name == 'split_indices':
                    continue
                elif model_name == "MAP":
                    normalized_posterior = hsea.bayesian_map_to_grids(
                        model_result['results']['training'].posterior_probabilities
                    )
                    ground_truth = theta_data.ground_truth
                    spikes = theta_data.spikes
                    spike_count_percentile = 66.67
                else:
                    smoothed_means = model_result['results']['training'].smoothed_mean \
                        + model_result['results']['validation'].smoothed_mean
                    smoothed_covs = model_result['results']['training'].smoothed_cov \
                        + model_result['results']['validation'].smoothed_cov
                    normalized_posterior = hsea.gaussians_to_grid(
                        smoothed_means,
                        smoothed_covs,
                        environment_size,
                        bin_size,
                        model_result['model']['marginal_position_dims']
                    )
                    ground_truth = [
                        theta_data.ground_truth[i] for i in rat_session_mod[name]['split_indices']['train']
                    ] + [
                        theta_data.ground_truth[i] for i in rat_session_mod[name]['split_indices']['validation']
                    ]
                    spikes = [
                        theta_data.spikes[i] for i in rat_session_mod[name]['split_indices']['train']
                    ] + [
                        theta_data.spikes[i] for i in rat_session_mod[name]['split_indices']['validation']
                    ]
                    spike_count_percentile = None
                peak_posterior = hsea.peak_posterior(
                    normalized_posterior,
                    environment_size,
                    ground_truth,
                    spikes,
                    spike_count_percentile
                )
                session_models.get(model_name, {
                    "Linear": [],
                    "Open"  : []
                })[track_type].append(peak_posterior)
        outputs.append(session_models)
    return outputs